# Lab 1 — Linear Algebra cho Machine Learning

**Thời lượng:** 75–90 phút  
**Mục tiêu:** vector/matrix shape, dot product, norm, transpose, matrix multiplication và linear model.

> Trước mỗi phép `@`, hãy ghi shape của hai toán hạng và output dự kiến.

In [ ]:
from pathlib import Path
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'single_feature_regression.csv').exists()
print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('Project root:', ROOT)

## 1. Scalar, vector, matrix và tensor

In [ ]:
scalar = np.array(3.5)
vector = np.array([1.0, 2.0, 3.0])
matrix = np.arange(12, dtype=float).reshape(4, 3)
tensor = np.zeros((2, 4, 3))
for name, value in [('scalar', scalar), ('vector', vector), ('matrix', matrix), ('tensor', tensor)]:
    print(f'{name:7s} shape={value.shape}, ndim={value.ndim}, size={value.size}')
assert matrix.shape == (4, 3)
assert tensor.ndim == 3

## 2. Dot product, norm và cosine similarity

In [ ]:
a = np.array([1.0, 2.0, -1.0])
b = np.array([3.0, 0.0, 4.0])
dot = a @ b
norm_a = np.linalg.norm(a)
norm_b = np.linalg.norm(b)
distance = np.linalg.norm(a - b)
cosine = dot / (norm_a * norm_b)
print('a @ b:', dot)
print('||a||2:', norm_a)
print('||b||2:', norm_b)
print('distance:', distance)
print('cosine:', cosine)
assert dot == -1.0
np.testing.assert_allclose(norm_a, np.sqrt(6))

## 3. Matrix multiplication và shape audit

In [ ]:
A = np.arange(15, dtype=float).reshape(5, 3)
v = np.array([1.0, -2.0, 0.5])
B = np.arange(6, dtype=float).reshape(3, 2)
Av = A @ v
AB = A @ B
elementwise = A * v
print('(5,3) @ (3,) ->', Av.shape)
print('(5,3) @ (3,2) ->', AB.shape)
print('(5,3) * (3,) ->', elementwise.shape)
assert Av.shape == (5,)
assert AB.shape == (5, 2)
assert elementwise.shape == (5, 3)

In [ ]:
try:
    A @ np.ones(5)
except ValueError as exc:
    print('Expected shape error:', exc)

## 4. Design matrix, bias và prediction

Mỗi row là một sample; mỗi column là một feature. Bias column cho phép intercept nằm trong cùng phép nhân matrix.

In [ ]:
X = np.array([[2.0, 1.0], [4.0, 3.0], [6.0, 2.0], [8.0, 5.0]])
X_bias = np.column_stack([np.ones(X.shape[0]), X])
theta = np.array([5.0, 2.0, -3.0])
predictions = X_bias @ theta
manual = np.array([5 + 2*x1 - 3*x2 for x1, x2 in X])
print('X:', X.shape)
print('X_bias:', X_bias.shape)
print('theta:', theta.shape)
print('predictions:', predictions.shape, predictions)
np.testing.assert_allclose(predictions, manual)

## 5. Residual, MSE và gradient shape

In [ ]:
y = np.array([7.0, 5.0, 12.0, 6.0])
residual = predictions - y
mse = np.mean(residual ** 2)
gradient = (2.0 / len(y)) * (X_bias.T @ residual)
print('residual:', residual.shape)
print('X_bias.T:', X_bias.T.shape)
print('gradient:', gradient.shape, gradient)
print('MSE:', mse)
assert gradient.shape == theta.shape

## 6. Least-squares line trên dữ liệu một feature

In [ ]:
frame = pd.read_csv(ROOT / 'data' / 'single_feature_regression.csv')
x = frame['x'].to_numpy(dtype=float)
y_line = frame['y'].to_numpy(dtype=float)
design = np.column_stack([np.ones(len(x)), x])
theta_lstsq = np.linalg.lstsq(design, y_line, rcond=None)[0]
fitted = design @ theta_lstsq
mse_line = np.mean((fitted - y_line) ** 2)
rmse_line = np.sqrt(mse_line)
r2_line = 1 - np.sum((fitted-y_line)**2) / np.sum((y_line-y_line.mean())**2)
print('theta:', theta_lstsq)
print('MSE:', mse_line, 'RMSE:', rmse_line, 'R2:', r2_line)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y_line, alpha=0.7, label='samples')
order = np.argsort(x)
ax.plot(x[order], fitted[order], color='black', label='least-squares line')
ax.set(title='Single-feature linear regression', xlabel='x', ylabel='y')
ax.legend()
fig.tight_layout()
plt.show()

## Exit ticket

1. Giải thích `X @ theta` bằng weighted sum của từng row.
2. Vì sao gradient phải cùng shape theta?
3. `X.T` làm gì với residual?
4. MSE và RMSE khác nhau về đơn vị thế nào?

In [ ]:
assert abs(theta_lstsq[0] - 4.11596242) < 1e-6
assert abs(theta_lstsq[1] - 3.20655686) < 1e-6
print('Lab 1 checks: PASS')